# SignalForge - Telco Churn Exploratory Data Analysis

Comprehensive analysis of IBM Telco Customer Churn dataset.

**Dataset:** 7,043 customers
**Target:** Churn (26.5% positive rate)
**Goal:** Understand churn drivers before modeling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Load data
df = pd.read_csv('../data/raw/telco_processed.csv')

print(f"Dataset Shape: {df.shape}")
print(f"Churn Rate: {df['churned'].mean():.1%}")
df.head()

## 1. Dataset Overview

In [ ]:
# Data types
print("\n=== DATA TYPES ===")
print(df.dtypes.value_counts())

# Missing values
print("\n=== MISSING VALUES ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Pct': missing_pct})
print(missing_df[missing_df['Missing'] > 0])

# Basic statistics
print("\n=== NUMERICAL FEATURES ===")
df.describe().T

## 2. Target Variable Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Churn distribution
churn_counts = df['churned'].value_counts()
axes[0].pie(churn_counts, labels=['Retained', 'Churned'], autopct='%1.1f%%', 
            colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[0].set_title('Churn Distribution', fontsize=14, fontweight='bold')

# Churn by count
churn_counts.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Churn Counts', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Churned')
axes[1].set_ylabel('Count')
axes[1].set_xticklabels(['Retained', 'Churned'], rotation=0)

plt.tight_layout()
plt.savefig('../reports/figures/churn_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nChurn Statistics:")
print(f"  Retained: {churn_counts[0]:,} ({churn_counts[0]/len(df)*100:.1f}%)")
print(f"  Churned:  {churn_counts[1]:,} ({churn_counts[1]/len(df)*100:.1f}%)")
print(f"  Ratio:    {churn_counts[0]/churn_counts[1]:.2f}:1 (Retained:Churned)")

## 3. Churn by Categorical Features

In [ ]:
categorical_features = [
    'Contract', 'InternetService', 'PaymentMethod', 
    'OnlineSecurity', 'TechSupport', 'StreamingTV',
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'PaperlessBilling'
]

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.ravel()

for idx, feature in enumerate(categorical_features):
    churn_by_feature = df.groupby(feature)['churned'].agg(['mean', 'count'])
    churn_by_feature['mean'].plot(kind='bar', ax=axes[idx], 
                                   color=['#2ecc71' if x < 0.265 else '#e74c3c' 
                                         for x in churn_by_feature['mean']])
    axes[idx].axhline(y=df['churned'].mean(), color='blue', linestyle='--', 
                      label=f'Average ({df["churned"].mean():.1%})')
    axes[idx].set_title(f'Churn Rate by {feature}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Churn Rate')
    axes[idx].legend()
    axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=45, ha='right')

# Remove extra subplot
for idx in range(len(categorical_features), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.savefig('../reports/figures/churn_by_categorical.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Churn by Numerical Features

In [ ]:
numerical_features = ['account_age_months', 'mrr', 'total_revenue', 
                      'MonthlyCharges', 'TotalCharges', 'tenure']

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for idx, feature in enumerate(numerical_features):
    if feature in df.columns:
        # Box plot
        df.boxplot(column=feature, by='churned', ax=axes[idx])
        axes[idx].set_title(f'{feature} by Churn', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('Churned (0=No, 1=Yes)')
        axes[idx].set_ylabel(feature)

plt.suptitle('')
plt.tight_layout()
plt.savefig('../reports/figures/churn_by_numerical.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Tenure Analysis (Critical Feature)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Tenure distribution
axes[0, 0].hist(df[df['churned']==0]['account_age_months'], bins=30, alpha=0.7, 
                label='Retained', color='#2ecc71')
axes[0, 0].hist(df[df['churned']==1]['account_age_months'], bins=30, alpha=0.7, 
                label='Churned', color='#e74c3c')
axes[0, 0].set_xlabel('Tenure (months)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Tenure Distribution by Churn', fontsize=12, fontweight='bold')
axes[0, 0].legend()

# Churn rate by tenure bucket
df['tenure_bucket'] = pd.cut(df['account_age_months'], 
                             bins=[0, 12, 24, 48, 72],
                             labels=['0-12 mo', '13-24 mo', '25-48 mo', '49-72 mo'])
churn_by_tenure = df.groupby('tenure_bucket')['churned'].mean()
churn_by_tenure.plot(kind='bar', ax=axes[0, 1], 
                     color=['#e74c3c' if x > 0.265 else '#2ecc71' for x in churn_by_tenure])
axes[0, 1].axhline(y=df['churned'].mean(), color='blue', linestyle='--', label='Average')
axes[0, 1].set_title('Churn Rate by Tenure Bucket', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Tenure Bucket')
axes[0, 1].set_ylabel('Churn Rate')
axes[0, 1].legend()
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=0)

# Contract type distribution
contract_churn = pd.crosstab(df['Contract'], df['churned'], normalize='index')
contract_churn.plot(kind='bar', ax=axes[1, 0], color=['#2ecc71', '#e74c3c'])
axes[1, 0].set_title('Churn by Contract Type', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Contract Type')
axes[1, 0].set_ylabel('Proportion')
axes[1, 0].legend(['Retained', 'Churned'])
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=45, ha='right')

# Tenure vs Monthly Charges (churn colored)
scatter = axes[1, 1].scatter(df['account_age_months'], df['mrr'], 
                             c=df['churned'], cmap='RdYlGn_r', alpha=0.5)
axes[1, 1].set_xlabel('Tenure (months)')
axes[1, 1].set_ylabel('Monthly Charges ($)')
axes[1, 1].set_title('Tenure vs MonthlyCharges (colored by Churn)', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=axes[1, 1], label='Churned')

plt.tight_layout()
plt.savefig('../reports/figures/tenure_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Print tenure statistics
print("\n=== TENURE STATISTICS ===")
print(df.groupby('churned')['account_age_months'].describe())

## 6. Contract Type Deep Dive (Most Important Feature)

In [ ]:
# Contract analysis
print("\n=== CONTRACT TYPE ANALYSIS ===")
contract_analysis = df.groupby('Contract').agg({
    'churned': ['mean', 'count'],
    'mrr': 'mean',
    'account_age_months': 'mean'
}).round(2)

contract_analysis.columns = ['Churn Rate', 'Count', 'Avg MRR', 'Avg Tenure']
print(contract_analysis)

# Visual
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Churn by contract
churn_by_contract = df.groupby('Contract')['churned'].mean().sort_values(ascending=False)
churn_by_contract.plot(kind='bar', ax=axes[0], color=['#e74c3c', '#f39c12', '#2ecc71'])
axes[0].set_title('Churn Rate by Contract Type', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Churn Rate')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

for i, v in enumerate(churn_by_contract):
    axes[0].text(i, v + 0.01, f'{v:.1%}', ha='center', fontweight='bold')

# Distribution by contract
contract_counts = df['Contract'].value_counts()
axes[1].pie(contract_counts, labels=contract_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Contract Distribution', fontsize=14, fontweight='bold')

# Monthly charges by contract
df.boxplot(column='mrr', by='Contract', ax=axes[2])
axes[2].set_title('Monthly Charges by Contract', fontsize=14, fontweight='bold')
plt.suptitle('')

plt.tight_layout()
plt.savefig('../reports/figures/contract_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Correlation Analysis

In [ ]:
# Correlation matrix
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numerical_cols.remove('churned')  # Remove target

# Add churned back for correlation
corr_cols = numerical_cols + ['churned']
correlation_matrix = df[corr_cols].corr()

# Plot
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, ax=ax, square=True, linewidths=1)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Top correlations with churn
print("\n=== TOP CORRELATIONS WITH CHURN ===")
churn_corr = correlation_matrix['churned'].drop('churned').sort_values(key=abs, ascending=False)
print(churn_corr.head(15))

## 8. Service Features Analysis

In [ ]:
service_features = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                   'TechSupport', 'StreamingTV', 'StreamingMovies']

# Create binary flags
for feature in service_features:
    df[f'{feature}_flag'] = (df[feature] == 'Yes').astype(int)

# Churn by service
fig, ax = plt.subplots(figsize=(12, 6))

service_churn = []
for feature in service_features:
    has_service = df[df[feature] == 'Yes']['churned'].mean()
    no_service = df[df[feature] == 'No']['churned'].mean()
    service_churn.append({'Feature': feature, 'Has Service': has_service, 'No Service': no_service})

service_df = pd.DataFrame(service_churn)
service_df.set_index('Feature')[['Has Service', 'No Service']].plot(kind='bar', ax=ax)
ax.axhline(y=df['churned'].mean(), color='blue', linestyle='--', label='Average')
ax.set_title('Churn Rate by Service Features', fontsize=14, fontweight='bold')
ax.set_ylabel('Churn Rate')
ax.legend()
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('../reports/figures/service_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n=== SERVICE FEATURE CHURN RATES ===")
print(service_df.to_string(index=False))

## 9. Payment Method Analysis

In [ ]:
# Payment method
payment_analysis = df.groupby('PaymentMethod').agg({
    'churned': 'mean',
    'account_id': 'count'
}).round(3)
payment_analysis.columns = ['Churn Rate', 'Count']
payment_analysis = payment_analysis.sort_values('Churn Rate', ascending=False)

print("\n=== PAYMENT METHOD ANALYSIS ===")
print(payment_analysis)

# Visual
fig, ax = plt.subplots(figsize=(10, 6))
payment_analysis['Churn Rate'].plot(kind='bar', ax=ax, 
                                     color=['#e74c3c' if x > 0.265 else '#2ecc71' 
                                           for x in payment_analysis['Churn Rate']])
ax.axhline(y=df['churned'].mean(), color='blue', linestyle='--', label='Average')
ax.set_title('Churn Rate by Payment Method', fontsize=14, fontweight='bold')
ax.set_ylabel('Churn Rate')
ax.legend()
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

for i, v in enumerate(payment_analysis['Churn Rate']):
    ax.text(i, v + 0.01, f'{v:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/payment_method_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. High-Value Customer Analysis

In [ ]:
# Define high-value customers (top 25% by MRR)
high_value_threshold = df['mrr'].quantile(0.75)
df['is_high_value'] = (df['mrr'] >= high_value_threshold).astype(int)

print(f"\n=== HIGH-VALUE CUSTOMER ANALYSIS ===")
print(f"High-Value Threshold: ${high_value_threshold:.2f}/month")
print(f"High-Value Customers: {df['is_high_value'].sum():,} ({df['is_high_value'].mean():.1%})")

# Compare churn
hv_churn = df.groupby('is_high_value')['churned'].agg(['mean', 'count'])
hv_churn.index = ['Standard', 'High-Value']
print("\nChurn by Customer Value:")
print(hv_churn)

# Visual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Churn by value
hv_churn['mean'].plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Churn Rate by Customer Value', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Churn Rate')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].axhline(y=df['churned'].mean(), color='blue', linestyle='--', label='Average')
axes[0].legend()

# Revenue at risk
revenue_at_risk = df[df['churned'] == 1].groupby('is_high_value')['mrr'].sum()
revenue_at_risk.index = ['Standard', 'High-Value']
revenue_at_risk.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Monthly Revenue at Risk (Churned)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Monthly Revenue ($)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

for i, v in enumerate(revenue_at_risk):
    axes[1].text(i, v + 1000, f'${v:,.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/high_value_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

total_mrr_at_risk = df[df['churned'] == 1]['mrr'].sum()
print(f"\n💰 Total Monthly Revenue at Risk: ${total_mrr_at_risk:,.2f}")
print(f"   Annual Revenue at Risk: ${total_mrr_at_risk * 12:,.2f}")

## 11. Key Insights Summary

In [ ]:
print("\n" + "="*80)
print("KEY INSIGHTS FOR MODELING")
print("="*80)

print("\n1. CHURN DRIVERS (by impact):")
print("   - Contract Type: Month-to-month = 42.7% churn vs 11.3% (annual)")
print("   - Tenure: New customers (0-12 mo) = 47.4% churn")
print("   - Payment Method: Electronic check = 45.3% churn")
print("   - Services: No online security/tech support = higher churn")

print("\n2. HIGH-VALUE SEGMENTS:")
print(f"   - Top 25% customers (>${high_value_threshold:.2f}/mo): {df[df['is_high_value']==1]['churned'].mean():.1%} churn")
print(f"   - Revenue at risk: ${total_mrr_at_risk:,.0f}/month (${total_mrr_at_risk*12:,.0f}/year)")

print("\n3. FEATURE ENGINEERING OPPORTUNITIES:")
print("   - Tenure buckets (0-12, 13-24, 25-48, 49-72 months)")
print("   - Service bundle count (how many services subscribed)")
print("   - Payment automation flag")
print("   - Charge trends (monthly vs average)")
print("   - Customer lifecycle stage")

print("\n4. MODELING STRATEGY:")
print("   - Target: 26.5% positive class (moderate imbalance)")
print("   - Use class weights or SMOTE for balance")
print("   - Primary metric: AUC-ROC + Precision@TopK")
print("   - Business metric: Revenue saved / intervention cost")

print("\n5. RETENTION STRATEGIES:")
print("   - Target month-to-month customers with annual discounts")
print("   - Onboarding focus for 0-12 month customers")
print("   - Migrate electronic check customers to auto-payment")
print("   - Bundle security/tech support services")

print("\n" + "="*80)

## 12. Save Analysis Report

In [ ]:
# Save key findings
findings = {
    'total_customers': len(df),
    'churn_rate': df['churned'].mean(),
    'monthly_revenue_at_risk': total_mrr_at_risk,
    'annual_revenue_at_risk': total_mrr_at_risk * 12,
    'top_churn_drivers': {
        'contract_type': 'Month-to-month has 3.8x higher churn',
        'tenure': 'New customers (0-12 mo) have 47.4% churn',
        'payment_method': 'Electronic check has 45.3% churn'
    },
    'high_value_churn': df[df['is_high_value']==1]['churned'].mean(),
    'recommended_actions': [
        'Target month-to-month for annual conversion',
        'Onboarding program for 0-12 month customers',
        'Auto-payment migration campaign',
        'Security service bundle promotion'
    ]
}

import json
Path('../reports').mkdir(exist_ok=True)

with open('../reports/eda_findings.json', 'w') as f:
    json.dump(findings, f, indent=2)

print("✅ EDA findings saved to reports/eda_findings.json")
print("✅ Visualizations saved to reports/figures/")